# Lily 1.5b v0.3 Comprehensive Evaluation — Modal A100 & L4
**Evaluates distilled Lily-1.5b-v0.3 across 6 major academic benchmarks**

Runs `lm-evaluation-harness` across `hellaswag`, `arc_challenge`, `mmlu`, `mmlu_redux_generative`, `gsm8k`, and `ifeval` on `abhinav0231/Lily-1.5b-v0.3` using Modal A100/L4 GPU with BFloat16 precision.

## Cell 1 — Install Dependencies (lm-evaluation-harness)

In [ ]:
# ==============================================================================
# Cell 1 — Install lm-evaluation-harness & Hugging Face Dependencies
# ==============================================================================
%uv pip install lm_eval[hf] datasets huggingface_hub wandb -q

## Cell 2 — Hardware Probe & Environment Check

In [ ]:
# ==============================================================================
# Cell 2 — Hardware & CUDA Device Property Probe
# ==============================================================================
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Version    : {torch.version.cuda}")

p = torch.cuda.get_device_properties(0)
print(f"\nGPU Device     : {p.name}")
print(f"VRAM Capacity  : {p.total_memory / 1e9:.1f} GB")
print(f"Compute        : cc={p.major}.{p.minor}")
print(f"BFloat16       : {'Supported' if p.major >= 8 else 'NOT supported'}")

assert torch.cuda.is_available(), "No GPU detected!"
assert p.major >= 8, f"Requires Ampere+ GPU (A100/L4/H100). Got cc={p.major}.{p.minor}"
print("\n✅ Hardware check passed")

## Cell 3 — Evaluation Configuration & Parameters

In [ ]:
# ==============================================================================
# Robust Authentication (Hugging Face & Weights & Biases)
# ==============================================================================
import os
from huggingface_hub import login, HfFolder

# Hugging Face Authentication
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = HfFolder.get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    try:
        login(token=HF_TOKEN)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")

# WandB Authentication
try:
    import wandb
    WANDB_TOKEN = os.environ.get("WANDB_API_KEY", "")
    if WANDB_TOKEN and WANDB_TOKEN != "YOUR_WANDB_KEY_HERE":
        wandb.login(key=WANDB_TOKEN, relogin=True)
        os.environ["WANDB_API_KEY"] = WANDB_TOKEN
        print("✅ Authenticated with Weights & Biases")
    else:
        print("ℹ️ WANDB_API_KEY not found. WandB tracking will operate in offline/disabled mode.")
        os.environ["WANDB_DISABLED"] = "true"
except ImportError:
    print("ℹ️ WandB module not installed. Operating without WandB tracking.")
    os.environ["WANDB_DISABLED"] = "true"


## Cell 4 — Launch Benchmark Evaluation Subprocess

In [ ]:
# ==============================================================================
# Cell 4 — Execute lm-evaluation-harness CLI Subprocess
# ==============================================================================
import subprocess

# Construct CLI command for multi-task evaluation
cmd = [
    "lm_eval",
    "--model", "hf",
    "--model_args", f"pretrained={MODEL_REPO},dtype=bfloat16,trust_remote_code=True,attn_implementation=sdpa",
    "--tasks", EVAL_TASKS,
    "--batch_size", str(BATCH_SIZE),
    "--apply_chat_template",
    "--output_path", OUTPUT_DIR,
    "--device", "cuda:0"
]

print(f"Launching benchmark evaluation suite...")
print(f"CMD: {' '.join(cmd)}\n")

res = subprocess.run(cmd, check=True)
print(f"\n✅ Benchmark evaluation finished with exit code {res.returncode}")

## Cell 5 — Display & Parse Evaluation Accuracy Summary

In [ ]:
# ==============================================================================
# Cell 5 — Parse Results JSON & Display Accuracy Metrics Across Benchmark Tasks
# ==============================================================================
import glob, json

result_files = glob.glob(f"{OUTPUT_DIR}/**/*.json", recursive=True)
if result_files:
    latest_file = sorted(result_files)[-1]
    print(f"Parsing evaluation results file: {latest_file}\n")
    try:
        data = json.load(open(latest_file))
        results = data.get("results", {})
        print("="*60)
        print(f"  BENCHMARK ACCURACY SUMMARY — {MODEL_REPO}")
        print("="*60)
        for task, metrics in results.items():
            acc = metrics.get("acc,none", metrics.get("acc", metrics.get("exact_match,none", "N/A")))
            if isinstance(acc, float):
                print(f"  {task:<25} : {acc*100:.2f}%")
            else:
                print(f"  {task:<25} : {acc}")
        print("="*60)
    except Exception as e:
        print(f"Could not parse results JSON: {e}")
else:
    print(f"Results saved in folder: {OUTPUT_DIR}")